# Waterbirds Last-Layer Finetuning With Sparse Probe

This notebook follows the Waterbirds setup from `zero-shot.ipynb`, but trains only the final ResNet-50 classification layer on a sparse probe set selected by `probes/probe_pred_loss_entropy.json`.

The selected probe indices are used for training. Group labels are used only for validation/test reporting.

In [1]:
import copy
import json
import os
import random
import sys
from pathlib import Path
from collections import Counter

# Make imports work whether the notebook is launched from repo root or this folder.
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / 'intermediate_gen').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print('repo_root:', repo_root)

import gdown
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.models import resnet50
from tqdm.auto import tqdm

from intermediate_gen.datasets import MyWaterBirdsDataset


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

SEED = 0
set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

repo_root: /home/hungnt/hungnt/intermediate-layer-generalization


/home/hungnt/anaconda3/envs/gmoe/lib/python3.9/site-packages/gdown/__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/hungnt/anaconda3/envs/gmoe/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda


In [2]:
DATA_ROOT = repo_root / 'data/waterbirds/waterbird_complete95_forest2water2'
PROBE_JSON = repo_root / 'probes/probe_pred_loss_entropy.json'
MODEL_PATH = repo_root / 'checkpoints/final_checkpoint.pt'
OUTPUT_PATH = repo_root / 'checkpoints/resnet50_waterbirds_sparse_probe_last_layer.pth'

DATA_ROOT = str(DATA_ROOT)
PROBE_JSON = str(PROBE_JSON)
MODEL_PATH = str(MODEL_PATH)
OUTPUT_PATH = str(OUTPUT_PATH)

BATCH_SIZE = 64
NUM_WORKERS = 4
EPOCHS = 50
LR = 1e-3
WEIGHT_DECAY = 1e-4

os.makedirs('checkpoints', exist_ok=True)

In [3]:
img_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

base_dataset = MyWaterBirdsDataset(
    DATA_ROOT,
    remove_minority_groups=False,
    transform=img_transform,
)

train_dataset = Subset(base_dataset, base_dataset.train_idxs)
val_dataset = Subset(base_dataset, base_dataset.val_idxs)
test_dataset = Subset(base_dataset, base_dataset.test_idxs)

print('train:', len(train_dataset), 'val:', len(val_dataset), 'test:', len(test_dataset))
for name, ds in [('train', train_dataset), ('val', val_dataset), ('test', test_dataset)]:
    groups = base_dataset.group_array[ds.indices]
    print(name, 'group counts:', np.bincount(groups, minlength=base_dataset.n_groups))

11788
11788
Split set: train
y counts [3682 1113]
p counts [3554 1241]
group counts [3498  184   56 1057]
Split set: val
y counts [933 266]
p counts [600 599]
group counts [467 466 133 133]
Split set: test
y counts [4510 1284]
p counts [2897 2897]
group counts [2255 2255  642  642]
train: 4795 val: 1199 test: 5794
train group counts: [3498  184   56 1057]
val group counts: [467 466 133 133]
test group counts: [2255 2255  642  642]


In [4]:
with open(PROBE_JSON, 'r') as f:
    probe = json.load(f)

selected_indices = [int(i) for i in probe['selected_indices']]
selected_set = set(selected_indices)
train_index_set = set(int(i) for i in base_dataset.train_idxs)
assert selected_set.issubset(train_index_set)

selected_dataset = Subset(base_dataset, selected_indices)

selected_y = base_dataset.y_array[selected_indices]
selected_g = base_dataset.group_array[selected_indices]
selected_p = base_dataset.p_array[selected_indices]
print('probe json:', PROBE_JSON)
print('selected:', len(selected_indices))
print('selected class counts:', Counter(selected_y.tolist()))
print('selected group counts:', Counter(selected_g.tolist()))
print('selected conflict fraction overall:', float(np.mean(selected_p != selected_y)))
print('probe hparams:', probe.get('hparams', {}))

probe json: /home/hungnt/hungnt/intermediate-layer-generalization/probes/probe_pred_loss_entropy.json
selected: 238
selected class counts: Counter({0: 119, 1: 119})
selected group counts: Counter({3: 104, 1: 62, 0: 57, 2: 15})
selected conflict fraction overall: 0.3235294117647059
probe hparams: {'alpha_pred': 1.0, 'beta_margin': 0.0, 'candidate_pool_multiplier': 5, 'class_balance': 'equal', 'coverage_similarity': 'cosine', 'coverage_universe': 'candidate_pool', 'coverage_weight': 0.2, 'delta_router': 1.0, 'gamma_loss': 1.0, 'probe_percentage': 0.05, 'redundancy_weight': 0.1, 'score_normalization': 'zscore', 'seed': 0, 'selection_method': 'ed_sps', 'similarity_mode': 'cosine_clamped', 'use_coverage': True, 'use_group_labels_for_eval': True, 'use_group_labels_for_selection': False, 'use_redundancy': True}


In [5]:
def unpack(sample):
    (img, y, attr, idx), (group, _) = sample
    return (
        img,
        torch.as_tensor(y, dtype=torch.long),
        torch.as_tensor(attr, dtype=torch.long),
        torch.as_tensor(group, dtype=torch.long),
        torch.as_tensor(idx, dtype=torch.long),
    )


def collate(samples):
    imgs, ys, attrs, groups, idxs = zip(*[unpack(s) for s in samples])
    return torch.stack(imgs), torch.stack(ys), torch.stack(attrs), torch.stack(groups), torch.stack(idxs)

probe_loader = DataLoader(
    selected_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate,
)

In [6]:
assert os.path.exists(MODEL_PATH), f'Missing checkpoint: {MODEL_PATH}'
print('loading checkpoint:', MODEL_PATH)

model = resnet50(pretrained=False, num_classes=2)
ckpt = torch.load(MODEL_PATH, map_location='cpu')
if isinstance(ckpt, dict):
    for key in ['model_state_dict', 'state_dict', 'model', 'net']:
        if key in ckpt and isinstance(ckpt[key], dict):
            ckpt = ckpt[key]
            print('using checkpoint key:', key)
            break

# Handle DataParallel-style prefixes if present.
if isinstance(ckpt, dict) and any(k.startswith('module.') for k in ckpt.keys()):
    ckpt = {k.removeprefix('module.'): v for k, v in ckpt.items()}

missing, unexpected = model.load_state_dict(ckpt, strict=False)
print('missing keys:', missing)
print('unexpected keys:', unexpected)
model.to(device)

for p in model.parameters():
    p.requires_grad_(False)
for p in model.fc.parameters():
    p.requires_grad_(True)

print('trainable parameters:')
for name, p in model.named_parameters():
    if p.requires_grad:
        print(name, tuple(p.shape))


loading checkpoint: /home/hungnt/hungnt/intermediate-layer-generalization/checkpoints/final_checkpoint.pt
missing keys: []
unexpected keys: []


/home/hungnt/anaconda3/envs/gmoe/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/hungnt/anaconda3/envs/gmoe/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


trainable parameters:
fc.weight (2, 2048)
fc.bias (2,)


In [7]:
@torch.no_grad()
def evaluate(model, loader, n_groups):
    model.eval()
    total_loss = 0.0
    total = 0
    correct = 0
    g_correct = np.zeros(n_groups, dtype=np.int64)
    g_total = np.zeros(n_groups, dtype=np.int64)

    for x, y, attrs, groups, idxs in loader:
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        losses = F.cross_entropy(logits, y, reduction='none')
        preds = logits.argmax(dim=1)
        ok = preds.eq(y)

        total_loss += float(losses.sum().item())
        total += int(y.numel())
        correct += int(ok.sum().item())

        groups_np = groups.numpy()
        ok_np = ok.cpu().numpy()
        for g in range(n_groups):
            mask = groups_np == g
            g_total[g] += int(mask.sum())
            g_correct[g] += int(ok_np[mask].sum())

    per_group = g_correct / np.maximum(g_total, 1)
    return {
        'loss': total_loss / max(total, 1),
        'accuracy': correct / max(total, 1),
        'worst_group_accuracy': float(per_group.min()),
        'avg_group_accuracy': float(per_group.mean()),
        'per_group_accuracy': per_group.tolist(),
        'per_group_total': g_total.tolist(),
    }

print('Before last-layer finetuning')
print('VAL:', evaluate(model, val_loader, base_dataset.n_groups))
print('TEST:', evaluate(model, test_loader, base_dataset.n_groups))

Before last-layer finetuning
VAL: {'loss': 0.33853339313764785, 'accuracy': 0.9165971643035863, 'worst_group_accuracy': 0.706766917293233, 'avg_group_accuracy': 0.8899390493124486, 'per_group_accuracy': [0.9957173447537473, 0.8798283261802575, 0.706766917293233, 0.9774436090225563], 'per_group_total': [467, 466, 133, 133]}
TEST: {'loss': 0.3821716219865745, 'accuracy': 0.9180186399723852, 'worst_group_accuracy': 0.7383177570093458, 'avg_group_accuracy': 0.894137465376353, 'per_group_accuracy': [0.9951219512195122, 0.8789356984478935, 0.7383177570093458, 0.9641744548286605], 'per_group_total': [2255, 2255, 642, 642]}


In [8]:
optimizer = torch.optim.AdamW(model.fc.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

best_state = copy.deepcopy(model.fc.state_dict())
best_val_wga = -np.inf
history = []

for epoch in tqdm(range(EPOCHS)):
    # Keep the frozen backbone and BatchNorm statistics fixed; train only fc.
    model.eval()
    model.fc.train()
    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for x, y, attrs, groups, idxs in probe_loader:
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        loss = F.cross_entropy(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += float(loss.item()) * int(y.numel())
        running_correct += int(logits.argmax(1).eq(y).sum().item())
        running_total += int(y.numel())

    val_metrics = evaluate(model, val_loader, base_dataset.n_groups)
    train_loss = running_loss / max(running_total, 1)
    train_acc = running_correct / max(running_total, 1)
    record = {
        'epoch': epoch,
        'train_loss': train_loss,
        'train_accuracy': train_acc,
        **{f'val_{k}': v for k, v in val_metrics.items()},
    }
    history.append(record)

    print(
        f"epoch={epoch:03d} train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
        f"val_acc={val_metrics['accuracy']:.4f} val_wga={val_metrics['worst_group_accuracy']:.4f} "
        f"per_group={['%.3f' % x for x in val_metrics['per_group_accuracy']]}"
    )

    if val_metrics['worst_group_accuracy'] > best_val_wga:
        best_val_wga = val_metrics['worst_group_accuracy']
        best_state = copy.deepcopy(model.fc.state_dict())
        torch.save({
            'model_state': model.state_dict(),
            'fc_state': model.fc.state_dict(),
            'epoch': epoch,
            'val_metrics': val_metrics,
            'probe_json': PROBE_JSON,
            'selected_indices': selected_indices,
            'hparams': {
                'epochs': EPOCHS,
                'batch_size': BATCH_SIZE,
                'lr': LR,
                'weight_decay': WEIGHT_DECAY,
                'seed': SEED,
                'trainable': 'fc_only',
            },
        }, OUTPUT_PATH)
        print('  saved best:', OUTPUT_PATH)

  2%|▏         | 1/50 [00:01<01:14,  1.53s/it]

epoch=000 train_loss=0.3772 train_acc=0.9076 val_acc=0.9058 val_wga=0.7744 per_group=['0.994', '0.835', '0.774', '0.977']
  saved best: /home/hungnt/hungnt/intermediate-layer-generalization/checkpoints/resnet50_waterbirds_sparse_probe_last_layer.pth


  4%|▍         | 2/50 [00:02<01:11,  1.49s/it]

epoch=001 train_loss=0.3423 train_acc=0.9202 val_acc=0.9058 val_wga=0.7744 per_group=['0.994', '0.835', '0.774', '0.977']


  6%|▌         | 3/50 [00:04<01:09,  1.48s/it]

...

epoch=047 train_loss=0.0221 train_acc=1.0000 val_acc=0.9233 val_wga=0.8571 per_group=['0.985', '0.865', '0.857', '0.977']


 98%|█████████▊| 49/50 [01:12<00:01,  1.46s/it]

epoch=048 train_loss=0.0213 train_acc=1.0000 val_acc=0.9258 val_wga=0.8421 per_group=['0.987', '0.873', '0.842', '0.977']


100%|██████████| 50/50 [01:13<00:00,  1.47s/it]

epoch=049 train_loss=0.0208 train_acc=1.0000 val_acc=0.9266 val_wga=0.8571 per_group=['0.987', '0.871', '0.857', '0.977']


In [9]:
model.fc.load_state_dict(best_state)
print('Best validation WGA:', best_val_wga)
print('VAL:', evaluate(model, val_loader, base_dataset.n_groups))
print('TEST:', evaluate(model, test_loader, base_dataset.n_groups))
print('checkpoint:', OUTPUT_PATH)

Best validation WGA: 0.8646616541353384
VAL: {'loss': 0.18590698110947915, 'accuracy': 0.926605504587156, 'worst_group_accuracy': 0.8646616541353384, 'avg_group_accuracy': 0.9245890024663722, 'per_group_accuracy': [0.987152034261242, 0.869098712446352, 0.8646616541353384, 0.9774436090225563], 'per_group_total': [467, 466, 133, 133]}
TEST: {'loss': 0.20525987633022721, 'accuracy': 0.9301001035554022, 'worst_group_accuracy': 0.8473520249221184, 'avg_group_accuracy': 0.921953119063901, 'per_group_accuracy': [0.9866962305986696, 0.8864745011086475, 0.8473520249221184, 0.9672897196261683], 'per_group_total': [2255, 2255, 642, 642]}
checkpoint: /home/hungnt/hungnt/intermediate-layer-generalization/checkpoints/resnet50_waterbirds_sparse_probe_last_layer.pth


In [10]:
# Optional: inspect training history as a dataframe.
import pandas as pd

history_df = pd.DataFrame(history)
history_df.tail()

,epoch,train_loss,train_accuracy,val_loss,val_accuracy,val_worst_group_accuracy,val_avg_group_accuracy,val_per_group_accuracy,val_per_group_total
45,45,0.023119,1.0,0.205282,0.925771,0.857143,0.922709,"[0.987152034261242, 0.869098712446352, 0.85714...","[467, 466, 133, 133]"
46,46,0.022231,1.0,0.210226,0.923269,0.857143,0.921101,"[0.9850107066381156, 0.8648068669527897, 0.857...","[467, 466, 133, 133]"
47,47,0.022138,1.0,0.213172,0.923269,0.857143,0.921101,"[0.9850107066381156, 0.8648068669527897, 0.857...","[467, 466, 133, 133]"
48,48,0.021304,1.0,0.203312,0.925771,0.842105,0.920023,"[0.987152034261242, 0.8733905579399142, 0.8421...","[467, 466, 133, 133]"
49,49,0.020801,1.0,0.206130,0.926606,0.857143,0.923246,"[0.987152034261242, 0.871244635193133, 0.85714...","[467, 466, 133, 133]"
